In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Reinforcement learning on a language model

## Where this session fits

> **The limitation we start from: Part B ended with PPO, which needs a learned critic. On a language model, the critic is a second large network.**

- **The question:** how does a language model become a policy, and what does one step of RL on it involve and cost?
- **Position: Part C (language models), session 1 of 2**
    - Part A built the value-based path to DQN
    - Part B built the policy-gradient methods up to PPO
    - this session maps a language model onto the Part B loop and runs one PPO step on it
    - the next session, C2, is GRPO: the same loop without the critic
    - the Advanced course goes deeper: reward models, RLHF (reinforcement learning from human feedback), PPO and GRPO in depth, reasoning models
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. say what the state, action, episode and reward are for a language model, and why sampling makes it stochastic
2. name the four things that are new compared with the earlier environments
3. name who can produce a reward: a program, a learned reward model, a language-model judge
4. read one PPO step on a language model, including the per-token KL reward
5. say why supervised fine-tuning usually comes before RL


# The thesis: Part B applies, and four things are new

**Why this section?** Before the details, one claim that organizes Part C.

> **A language model is a policy over tokens. RL fine-tuning is the Part B policy gradient applied to it: the theorem is unchanged.**

| What is new on a language model | Consequence | Where |
|:--|:--|:--|
| 1. the action is a token from a huge vocabulary, the state is text | the policy must be a network, and we start from a model that already works | this session |
| 2. one reward for the whole completion, and restarting from a prompt costs only another sample | credit is coarse, and a **group** of $G$ completions from one prompt is cheap ($G$ is the group size) | this session, then C2 |
| 3. generating text is already the main cost, and a critic is a second large network | the critic is the costly addition: can the baseline come without it? | C2 |
| 4. the reward is often a proxy for what we want | the policy must stay near a model we trust (the KL reward) | this session |


# A language model is a policy

**Why this section?** Everything in Part B needs a policy $\pi_\theta(\act \mid \state)$, states, actions and episodes. A language model already has all four, once we name them.

> **A language model already is a policy: the prompt and the completion so far are the state, the next token is the action, and the whole completion is the episode.**

- a **token** is a piece of text from a fixed vocabulary $\Vocab$ (a word, part of a word, a punctuation mark)
- the prompt $\x = (\x_1, \ldots, \x_m)$ and the **completion** $\y = (\y_1, \ldots, \y_n)$, with $n = |\y|$ are both **sequences of tokens**
    - a completion is the model's response to a prompt
    - the model writes it one token at a time, each conditioned on everything before it

| Reinforcement learning | Language model |
|:--|:--|
| **state** $\stateseq_\tt$ | the prompt and the completion so far: $(\x, \y_{<\tt})$ |
| **action** $\actseq_\tt$ | the next token $\y_\tt$, chosen from $\Vocab$ |
| **policy** $\pi_\theta(\actseq_\tt \mid \stateseq_\tt)$ | the model's next-token distribution $\pi_\theta(\y_\tt \mid \x, \y_{<\tt})$ |
| **episode** | one whole completion, with $n$ steps (the episode length $|\traj|$ of earlier sessions) |


**A completion as an episode**

<img src="images/llm_episode.png" width=95%>

- the *state* is the prompt and the tokens generated so far
- the *action* is the next token
- the whole completion is one episode
- the reward usually arrives only at the end (sparse), as on Frozen Lake
- the episode ends when the model emits an end-of-completion token


**The first state, and the indexing**

- the first state is the prompt alone: no token has been chosen yet
- the state at step $\tt$ holds the $\tt - 1$ tokens before position $\tt$
    - the action taken in it is the token $\y_\tt$
    - some papers index the state by tokens already written, so the same step reads $\pi(\actseq_\tt \mid \stateseq_{\tt-1})$
    - it is the same step: we keep the course convention, with the action taken in $\stateseq_\tt$
- the next state is the old state with the chosen token appended


**The policy is a distribution, so the actor is stochastic**

> **A language model does not output a token: it outputs a probability for every token in $\Vocab$, and a token is then sampled from it.**

- the model computes a score $z_v$ (a **logit**) for every token $v$ in the vocabulary
    - the next-token probability is a softmax of the scores divided by a **temperature** $T$
    - $\pi_\theta(v \mid \x, \y_{<\tt}) \propto \exp(z_v / T)$
- temperature $T > 0$: the policy is stochastic, so the same prompt can give different completions
    - a lower $T$ concentrates the probability on the best-scoring tokens
    - a higher $T$ spreads it out
- $T \to 0$ always picks the highest-scoring token (greedy decoding): a deterministic policy
- **RL needs the stochastic case**
    - exploration comes from sampling: a completion that is never sampled can never be reinforced
    - the policy gradient uses $\log \pi_\theta$ of the sampled tokens, so we sample from the distribution the loss assumes (usually $T = 1$)


**One sampled completion is one trajectory**

- sampling a completion is **rolling out the policy**: a trajectory $\traj = (\stateseq_1, \actseq_1, \ldots)$ of Part B
    - the actions are the tokens $\y_1, \ldots, \y_n$
    - the states are the prefixes
- two samples for the same prompt are two different episodes that **start from the same state**
    - restarting is free: just sample again (used in C2)
- the transition is trivial: the next state is the old state plus the chosen token, with no randomness from an environment
    - so the probability of a whole completion is the product of the token probabilities
    - that product is why the Part B policy gradient carries over with $\log \pi_\theta$ summed over the tokens


**Why the policy must be a network**

- the state is huge and the action set is the whole vocabulary: a table is impossible
- the policy must be a network (Part A's limitation, Part B's answer)


## Why a reward, and not a label

**Why this section?** Before asking where a reward comes from, ask why we want one: supervised fine-tuning already trains a model on answers.

> **Supervised loss measures distance from one exact label, and is lowest at the label. A reward can rank every answer, including ones better than the label.**

- prompt: "How are you?"; the supervised target (the label) is "I am fine, thank you"

| Answer | Supervised loss | Reward (illustrative) |
|:--|:--|:--|
| The sky is blue | large | low |
| I am fine | large | medium |
| I am fine, thank you | **zero** | high |
| I am fine, thank you, and how are you? | large | **higher still** |

- the supervised loss is lowest only when the answer equals the label
    - it counts "I am fine" as wrong, though it is a good answer
    - it cannot prefer the last answer, which is better than the label
- a reward ranks the answers: more polite is better, and the exchange is richer
- this is why RL needs a **reward** and not only labels: someone, or something, must score answers the label does not contain

**What comes first, the idea or the formula**

- the table above is a preference before it is a number
    - "thank you" beats "I am fine"; the question beats both
- a reward turns such preferences into numbers a policy-gradient method can use
- the next slide asks who produces the numbers
- the reward values here are made up for the example
    - no real scorer is being quoted


## Where does the reward come from?

**Why this section?** A policy gradient needs a reward for each completion. Classify by who produces the label.

> **A reward can come from a program (a verifier), a learned reward model, or a language-model judge.**

| Label produced by | Example | Strength | Weakness |
|:--|:--|:--|:--|
| a **program** (a **verifier**): RLVR | the final number of a math problem; unit tests for code | exact on what it checks, cheap, no human label per completion (but someone writes the checker, and it can accept wrong answers) | only for checkable tasks |
| a **learned model** (a **reward model**) | trained on human comparisons of chat answers (next slide) | covers open-ended tasks | an imperfect proxy |
| a **language model as judge** | scores or ranks completions against a rubric | flexible, no training | has its own biases (for example, favoring long answers) |


**A verifier by example**

> **Using a verifier as the reward is called RLVR: reinforcement learning with verifiable rewards.**

- prompt: "What is $17 \times 24$?"
- completion A ends "so the answer is 408": the verifier gives reward $1$
- completion B ends "so the answer is 418": reward $0$
- the reward arrives once, after the last token
    - the verifier knows nothing about the steps before it
- a verifier checks only what its author wrote down
    - a completion that states 408 with no working at all also scores $1$


**A reward model learns from comparisons (a gloss)**

> **When no program can score a completion, a reward model learns a score from comparisons. Nothing else in this series needs preference data.**

- people can often say which of two completions is better, even when they cannot score one
- **preference data** records comparisons: one example is a triple $(\x, \y^+, \y^-)$
    - a prompt, a preferred completion, a non-preferred one
- the **Bradley-Terry model** turns two scores into a probability that one completion is preferred
    - $P(\y^+ \succ \y^-) = \sigma\bigl(\rew_\psi(\x, \y^+) - \rew_\psi(\x, \y^-)\bigr)$
    - $\succ$ reads "is preferred to"
    - the **sigmoid** $\sigma$ turns the score difference into a probability between $0$ and $1$
    - equal scores give $\frac{1}{2}$, and a larger gap in favor of $\y^+$ pushes it toward $1$
- it is a **classifier**: given the pair, predict which completion won
    - fitting $\psi$ is ordinary classification on many such pairs
- fitting the **reward model** $\rew_\psi$ (parameters $\psi$, not the critic's $\phi$) to many comparisons gives one score per completion
- that score is the same kind of reward a verifier gives
- the data, the training and the full pipeline, **RLHF** (reinforcement learning from human feedback), are in the Advanced course


**The reward is only a proxy**

- the reward is only a proxy for what we want, so the policy can learn to exploit it
    - **reward hacking** is getting a high reward without the quality the reward was meant to measure
    - Microduck in the A0 tour found an *unexpected strategy*
    - that counts as hacking only if the reward is earned in a way the designers did not want
    - for example, a model trained against a reward model may learn long, confident answers that score well but are not better
- a verifier is a proxy too: it checks only what its author wrote down


## One PPO step on a language model

**Why this section?** This is the PPO loop of Part B (B4), with the language-model names filled in.

> **One PPO step: sample completions, score them, turn the scores into per-token advantages with a critic, and take a clipped step.**

- **Sample:** for a prompt $\x$, draw completions $\y$ from the current policy $\pi_\theta$ (it becomes the old policy)
- **Score:** each completion gets one reward $\rew(\x, \y)$, from a verifier, a reward model or a judge
- **Credit:** a critic $\statevalfun_\phi$ estimates the value of every prefix, and GAE (B3) turns that into an advantage for every token
- **Update:** a clipped policy-gradient step (B4), repeated for $K$ epochs on the same batch
    - it raises the probability of tokens with a positive advantage and lowers it for the rest
- repeat with fresh samples


**Credit: one score becomes a weight for each token**

> **A token is judged by what it changed, not by the final score alone: a good start in a bad completion is blamed less than the tokens after it.**

<img src="images/c1_credit.png" width=70%>

- the reward is one number at the end, but the policy gradient needs a weight for every token
- the critic's value of each prefix is its estimate of the final reward from there
- the **surprise** is how much better or worse a token's step went than the critic expected
- the advantage of a token adds its surprise and all later surprises
- in the example, token 1 earned $+0.4$, but the later surprises ($-0.7$) bring its advantage to $-0.3$
    - tokens 2 to 4 get $-0.7$
    - without a critic, all four tokens would get the same weight (C2)

**How the surprise is computed**

- the **surprise** $\delta_\tt$ (B3) is the reward, plus the value of the next prefix, minus the value of this one
    - reward plus next value is the one-step estimate of this prefix's value (the TD target of A3 and B3, here with $\disc = 1$)
    - the surprise is that estimate minus the critic's current value
- each token uses **its own** reward: $0$ for tokens 1 to 3, the score for token 4
    - token 1: $0 + 0.9 - 0.5 = +0.4$
- positive: the token left the completion better placed than the critic expected
- negative: worse
- the advantage adds a token's surprise and all later ones (GAE with $\lambda = 1$, B3)
- the value after the last token is $0$, because the completion has ended
- the example leaves out the KL term, and uses $\disc = 1$


**The models a PPO step touches**

<img src="images/c1_four_models.png" width=90%>

- **actor** and **critic** are trained
- the **old actor** is a snapshot (only its stored log-probabilities are needed)
- the **reference** is a frozen copy of the model we started from (next slide)
- the critic scores every prefix of a long completion, and is often about as large as the actor


## The reference model and the KL reward

**Why this section?** The reward is a proxy (reward hacking), and the policy starts from a model we already like. We want to improve without wandering.

> **RLHF-style PPO puts the KL penalty in the reward: every token pays $\beta$ times the log-ratio of the policy to a frozen reference model.**

- the **reference model** $\pi_{\text{ref}}$ is a frozen copy of the model before RL (for example the supervised fine-tuned model)
- the reward at token $\tt$ has two parts
    - **the KL term, at every token:** $-\beta \log \dfrac{\pi_\theta(\y_\tt \mid \x, \y_{<\tt})}{\pi_{\text{ref}}(\y_\tt \mid \x, \y_{<\tt})}$
    - **the score $\rew(\x, \y)$, at the last token only**
- the KL term is zero when the policy and the reference agree on the token
- it is negative when the policy likes the token more than the reference does, and positive when it likes it less
- **KL divergence** (B4) measures how different two distributions are
- summed over a sampled completion, it estimates $\beta \, \KL(\pi_\theta \,\|\, \pi_{\text{ref}})$ for that completion

**The KL is a dense, per-step reward**

<img src="images/c1_token_rewards.png" width=85%>

- the score arrives once, at the end; the KL term arrives at every token
- so the critic and GAE of Part B handle it with no change: it is just a reward at each step
- large $\beta$: stay close to the reference; small $\beta$: chase the score
- it limits how far the policy can drift toward a hack, and tends to keep the text near the reference's fluency
- it does not prevent hacks that stay close to the reference

**Three different KLs in this series**

| Where | Between | Role |
|:--|:--|:--|
| the TRPO constraint (B4) | the old policy and the new policy | limits one update |
| the PPO reward (this session) | the current policy and the frozen reference | anchors the whole run, inside the reward |
| the GRPO loss term (C2) | the current policy and the frozen reference | the same anchor, as a loss term |

- clipping (B4) limits the move from the old policy to the new one within an iteration
- the reference KL anchors to one *fixed* model across the whole run
- it is not part of the original PPO paper; it comes from RLHF practice
- C2 puts the KL in the loss instead, which suits GRPO's one advantage per completion


## Why supervised fine-tuning usually comes before RL

**Why this section?** A condition for starting RL: the reward must be reachable.

> **RL raises the probability of completions the model already produces some of the time, so supervised fine-tuning usually comes first.**

- **supervised fine-tuning** (SFT) trains on example completions: every token has a target
    - RL has a reward instead of a target (A0's opening table)
- RL **raises the probability of completions the model already produces** some of the time and that earn reward
- if no sampled completion earns reward (a binary verifier reward), there is no signal
    - in GRPO (C2), a group with identical rewards (all failures, or all successes) gives advantage $0$
- so a model that rarely produces the required format is unlikely to find it by RL
    - SFT first puts that behavior within reach
- this is a condition, not a law
    - a strong base model may already succeed often enough to start RL directly (some recent reasoning models did)
- SFT also supplies the fluent reference model for the KL reward


**Before spending compute**

- a check: sample a group of $G$ completions for a few prompts
- many groups should contain at least one success (a *hit*) and at least one failure
    - with success chance $p$ per completion, a group contains a hit with probability $1 - (1-p)^G$
        - about $G p$ when that is small
        - for $p = 0.01$, $G = 8$: about $8\%$


## What this costs, and what can go wrong

**Why this section?** The costs that motivate the next session and the Advanced course.

> **The new cost is the critic: a second large network that scores every prefix of every completion. Next session: can the baseline come without it?**

- **the critic is the added cost:** a second large network to train, store and run, and its mistakes become bias (B3, B4)
- **generation is already the main cost:** many long completions per prompt, every iteration
- **the reward is a proxy:** hacking, judge biases, verifiers that accept wrong answers
- **sparse credit:** one reward for a long completion
- **no signal on easy or impossible prompts:** when every sampled completion gets the same reward


## Aside: the probability of a whole completion

- the policy factorizes over tokens: $\pi_\theta(\y \mid \x) = \prod_{\tt=1}^{|\y|} \pi_\theta(\y_\tt \mid \x, \y_{<\tt})$
- this is the probability of the *entire* episode
- the policy-gradient methods of Part B work token by token and do not need it
- methods that compare whole completions (in the Advanced course) do


# What we established, and what it sets up

- **Established**
    - Part B applies to a language model, and four things are new
        - a token action space, one reward per completion, costly generation, a proxy reward
    - a language model is a policy: state = prompt plus completion so far, action = next token, a completion is an episode
    - the reward comes from a program, a learned reward model or a judge: each a proxy that can be hacked
    - one PPO step: sample, score, per-token advantages from a critic, a clipped step
    - the KL penalty is a per-token reward to a frozen reference model, which keeps the policy from wandering
    - RL amplifies what the model already does, so supervised fine-tuning usually comes first
- **Sets up**
    - **the limitation:** the critic is a second large network
    - next session (C2): GRPO, a baseline from a group of completions, with no critic
    - the Advanced course: reward models and RLHF, PPO and GRPO in depth, reasoning models
